# Selecting Keywords

This notebook demonstrates extracting keywords from clusters using the `InformationWeightTransformer`.
As we will see, the information weight is particularly well suited to this task, and in fact it is already used to find keywords from documents clusters in the [Toponymy](https://github.com/TutteInstitute/toponymy) topic naming library.

For this example, our keywords will actually be key ingredients and the documents will be recipes, with clusters given by a ground truth cuisine.
The data was derived from the `What's Cooking?` Kaggle competition.

In [1]:
import warnings

# Suppress warnings
warnings.simplefilter("ignore")

In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from infoweight import InformationWeightTransformer

## Load the Data

The data is in `data/recipes.feather` and we have concatenated the ingredients column using a `"/"`.
We can one-hot encode the ingredients using scikit-learns's `CountVectorizer` with a custom tokenizer that splits on /.

In [3]:
df = pd.read_feather("data/recipes.feather")
df

,Cuisine,Ingredients
0,Greek,romaine lettuce/black olives/grape tomatoes/ga...
1,Southern Us,plain flour/ground pepper/salt/tomatoes/ground...
2,Filipino,eggs/pepper/salt/mayonaise/cooking oil/green c...
3,Indian,water/vegetable oil/wheat/salt
4,Indian,black pepper/shallots/cornflour/cayenne pepper...
...,...,...
39005,Irish,light brown sugar/granulated sugar/butter/warm...
39006,Italian,KRAFT Zesty Italian Dressing/purple onion/broc...
39007,Irish,eggs/citrus fruit/raisins/sourdough starter/fl...
39008,Chinese,boneless chicken skinless thigh/minced garlic/...


In [4]:
count_vectorizer = CountVectorizer(tokenizer=lambda x: x.split("/"), min_df=3)
counts = count_vectorizer.fit_transform(df["Ingredients"])
column_ingredients = (
    count_vectorizer.get_feature_names_out()
)  # Store the ingredient associated with each column
counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 420133 stored elements and shape (39010, 4108)>

## Just Count

A naive approach to keywords would be to simply count the number of recipes from a cuisine that contain each ingredients appears and take the top 3.

In [5]:
keywords = pd.DataFrame()
keywords["Cuisine"] = df["Cuisine"].unique()

most_common = []
for i in range(len(keywords)):
    cuisine = keywords.iloc[i].Cuisine
    row_in_cuisine = (df["Cuisine"] == cuisine).values
    ingredient_counts = np.asarray(counts[row_in_cuisine].sum(axis=0)).reshape(-1)
    top_ingredient_indices = np.argsort(ingredient_counts)[-3:][::-1]
    top_ingredients = column_ingredients[top_ingredient_indices]
    most_common.append(" / ".join(top_ingredients))
keywords["Most Common"] = most_common

with pd.option_context("max_colWidth", None):
    display(keywords)

,Cuisine,Most Common
0,Greek,salt / olive oil / dried oregano
1,Southern Us,salt / butter / all-purpose flour
2,Filipino,salt / garlic / onions
3,Indian,salt / onions / garam masala
4,Jamaican,salt / onions / water
5,Spanish,salt / olive oil / garlic cloves
6,Italian,salt / olive oil / garlic cloves
7,Mexican,salt / onions / ground cumin
8,Chinese,soy sauce / sesame oil / corn starch
9,British,salt / all-purpose flour / butter


That's a lot of salt!
These most frequent words are not very good at distinguishing the cuisines, because there are some ingredients that are just too universally common.

## Unsupervised

One idea is to first weight each term by how "informative" it is compared to all recipes.
This is the essential idea of IDF (the count is TF) and the information weight.
The weight $IDF(i) = \log (n/r_i)$, where $n$ is the total number of recipes and $r_i$ is the number of recipes than use ingredient $i$.
The information weight is a bit more complicated, and we refer to the forth coming `how it works` documentation page for details.

In [6]:
tfidf = TfidfTransformer(norm=None, smooth_idf=False)
iwt = InformationWeightTransformer()

tfidf_counts = tfidf.fit_transform(counts)
iwt_counts = iwt.fit_transform(counts).tocsr()

tfidf_most_common = []
iwt_most_common = []
for i in range(len(keywords)):
    cuisine = keywords.iloc[i].Cuisine
    row_in_cuisine = (df["Cuisine"] == cuisine).values

    ingredient_counts = np.asarray(tfidf_counts[row_in_cuisine].sum(axis=0)).reshape(-1)
    top_ingredient_indices = np.argsort(ingredient_counts)[-3:][::-1]
    top_ingredients = column_ingredients[top_ingredient_indices]
    tfidf_most_common.append(" / ".join(top_ingredients))

    ingredient_counts = np.asarray(iwt_counts[row_in_cuisine].sum(axis=0)).reshape(-1)
    top_ingredient_indices = np.argsort(ingredient_counts)[-3:][::-1]
    top_ingredients = column_ingredients[top_ingredient_indices]
    iwt_most_common.append(r" / ".join(top_ingredients))

keywords["TFIDF"] = tfidf_most_common
keywords["IWT"] = iwt_most_common

with pd.option_context("max_colWidth", None):
    display(keywords)

,Cuisine,Most Common,TFIDF,IWT
0,Greek,salt / olive oil / dried oregano,feta cheese crumbles / olive oil / feta cheese,feta cheese crumbles / feta cheese / dried oregano
1,Southern Us,salt / butter / all-purpose flour,salt / butter / all-purpose flour,butter / all-purpose flour / buttermilk
2,Filipino,salt / garlic / onions,soy sauce / garlic / water,soy sauce / garlic / water
3,Indian,salt / onions / garam masala,garam masala / ground turmeric / salt,garam masala / ground turmeric / cumin seed
4,Jamaican,salt / onions / water,ground allspice / salt / dried thyme,ground allspice / dried thyme / scallions
5,Spanish,salt / olive oil / garlic cloves,olive oil / extra-virgin olive oil / garlic cloves,extra-virgin olive oil / olive oil / garlic cloves
6,Italian,salt / olive oil / garlic cloves,olive oil / grated parmesan cheese / salt,grated parmesan cheese / olive oil / extra-virgin olive oil
7,Mexican,salt / onions / ground cumin,ground cumin / chili powder / salt,avocado / jalapeno chilies / salsa
8,Chinese,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch
9,British,salt / all-purpose flour / butter,milk / all-purpose flour / salt,milk / unsalted butter / all-purpose flour


That's definitely better, but we can see that TFIDF kept a fair amount of salt.
The information weight has removed all the salt, but has kept more than a couple butter and fish sauce.

## Supervised

A further idea is to make use of the labels when determining how informative an ingredient is.
Instead of looking at the count matrix, we collapse each cuisines to a single row, i.e. one row per cuisine.
Applying TFIDF to this reduced matrix is called `c-TFIDIF`, the c stands for class based.
Let's try it with TFIDF and the information weight.

In [7]:
class_based_counts = np.empty((len(keywords), counts.shape[1]), dtype=counts.dtype)
for i in range(len(keywords)):
    cuisine = keywords.iloc[i].Cuisine
    row_in_cuisine = (df["Cuisine"] == cuisine).values
    in_cuisine_counts = np.asarray(counts[row_in_cuisine].sum(axis=0)).reshape(-1)
    class_based_counts[i] = in_cuisine_counts
class_based_counts = sp.csr_matrix(class_based_counts)
class_based_counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 25882 stored elements and shape (20, 4108)>

In [8]:
tfidf = TfidfTransformer()
weights = tfidf.fit_transform(class_based_counts)
top_ingredient_indices = np.argsort(weights.toarray(), axis=1)[:, -3:][:, ::-1]
top_ingredients = column_ingredients[top_ingredient_indices]
keywords["c-TFIDF"] = [" / ".join(i) for i in top_ingredients]

In [9]:
iwt = InformationWeightTransformer()
weights = iwt.fit_transform(class_based_counts)
top_ingredient_indices = np.argsort(weights.toarray(), axis=1)[:, -3:][:, ::-1]
top_ingredients = column_ingredients[top_ingredient_indices]
keywords["c-IWT"] = [" / ".join(i) for i in top_ingredients]

In [10]:
with pd.option_context("max_colWidth", None):
    display(keywords)

,Cuisine,Most Common,TFIDF,IWT,c-TFIDF,c-IWT
0,Greek,salt / olive oil / dried oregano,feta cheese crumbles / olive oil / feta cheese,feta cheese crumbles / feta cheese / dried oregano,salt / olive oil / feta cheese crumbles,feta cheese crumbles / feta cheese / dried oregano
1,Southern Us,salt / butter / all-purpose flour,salt / butter / all-purpose flour,butter / all-purpose flour / buttermilk,salt / butter / all-purpose flour,buttermilk / baking powder / all-purpose flour
2,Filipino,salt / garlic / onions,soy sauce / garlic / water,soy sauce / garlic / water,salt / garlic / onions,soy sauce / fish sauce / coconut milk
3,Indian,salt / onions / garam masala,garam masala / ground turmeric / salt,garam masala / ground turmeric / cumin seed,salt / garam masala / onions,garam masala / ground turmeric / cumin seed
4,Jamaican,salt / onions / water,ground allspice / salt / dried thyme,ground allspice / dried thyme / scallions,salt / onions / water,ground allspice / scotch bonnet chile / coconut milk
5,Spanish,salt / olive oil / garlic cloves,olive oil / extra-virgin olive oil / garlic cloves,extra-virgin olive oil / olive oil / garlic cloves,salt / olive oil / garlic cloves,saffron threads / extra-virgin olive oil / spanish chorizo
6,Italian,salt / olive oil / garlic cloves,olive oil / grated parmesan cheese / salt,grated parmesan cheese / olive oil / extra-virgin olive oil,salt / olive oil / grated parmesan cheese,grated parmesan cheese / olive oil / extra-virgin olive oil
7,Mexican,salt / onions / ground cumin,ground cumin / chili powder / salt,avocado / jalapeno chilies / salsa,salt / onions / ground cumin,corn tortillas / salsa / avocado
8,Chinese,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch
9,British,salt / all-purpose flour / butter,milk / all-purpose flour / salt,milk / unsalted butter / all-purpose flour,salt / all-purpose flour / butter,unsalted butter / all-purpose flour / milk


c-TFIDF does not seem to have done very well, but c-IWT looks great.
For example, c-IWT picks corn tortillas, salsa, and avocado for Mexican; a selection I would certainly agree with.

## 1-vs-all Supervised

There is one more idea, and this time it's specific to the information weight.
One probable problem with c-TFIDF is that the maximum information of a term is bounded by $\log c$ (recall there are only c rows in the reduced matrix).
However, since the information weight does not have this problem, we can reduce the matrix even further.

Instead of constructing a single reduced matrix with $c$ rows, for each class we'll create a reduced matrix with only two rows: one for the class and one for everything else.
The information weight rewards deviation from the marginal distribution, and since almost all the weight will be in the negative class, ingredients that appear mostly in the positive class will have large weight.
This is essentially parallel to training a 1-vs-all classifier, but we only get positive association as we don't have negative columns (i.e. not salt).

In [11]:
iwt_one_v_all = []
iwt = InformationWeightTransformer()
for i in range(len(keywords)):
    cuisine = keywords.iloc[i].Cuisine
    # This is an array with length = n_rows, and the entry is True if the row
    # is part of the current cuisine
    row_in_cuisine = (df["Cuisine"] == cuisine).values
    iwt.fit(counts, row_in_cuisine)  # pass the class labels as the second parameter
    top_ingredient_indices = np.argsort(iwt.information_weights_)[-3:][::-1]
    top_ingredients = column_ingredients[top_ingredient_indices]
    iwt_one_v_all.append(" / ".join(top_ingredients))
keywords["IWT 1-vs-all"] = iwt_one_v_all

with pd.option_context("max_colWidth", None):
    display(keywords)

,Cuisine,Most Common,TFIDF,IWT,c-TFIDF,c-IWT,IWT 1-vs-all
0,Greek,salt / olive oil / dried oregano,feta cheese crumbles / olive oil / feta cheese,feta cheese crumbles / feta cheese / dried oregano,salt / olive oil / feta cheese crumbles,feta cheese crumbles / feta cheese / dried oregano,greek feta / pita wedges / greek-style vinaigrette
1,Southern Us,salt / butter / all-purpose flour,salt / butter / all-purpose flour,butter / all-purpose flour / buttermilk,salt / butter / all-purpose flour,buttermilk / baking powder / all-purpose flour,nonfat vanilla frozen yogurt / smoked pork neck bones / butter crackers
2,Filipino,salt / garlic / onions,soy sauce / garlic / water,soy sauce / garlic / water,salt / garlic / onions,soy sauce / fish sauce / coconut milk,kangkong / milkfish / annatto powder
3,Indian,salt / onions / garam masala,garam masala / ground turmeric / salt,garam masala / ground turmeric / cumin seed,salt / garam masala / onions,garam masala / ground turmeric / cumin seed,full-fat plain yogurt / tandoori paste / low fat plain yoghurt
4,Jamaican,salt / onions / water,ground allspice / salt / dried thyme,ground allspice / dried thyme / scallions,salt / onions / water,ground allspice / scotch bonnet chile / coconut milk,cho-cho / jamaican jerk spice / caribbean jerk seasoning
5,Spanish,salt / olive oil / garlic cloves,olive oil / extra-virgin olive oil / garlic cloves,extra-virgin olive oil / olive oil / garlic cloves,salt / olive oil / garlic cloves,saffron threads / extra-virgin olive oil / spanish chorizo,low sodium vegetable juice / paella rice / rioja
6,Italian,salt / olive oil / garlic cloves,olive oil / grated parmesan cheese / salt,grated parmesan cheese / olive oil / extra-virgin olive oil,salt / olive oil / grated parmesan cheese,grated parmesan cheese / olive oil / extra-virgin olive oil,good seasons italian dressing mix / prego fresh mushroom italian sauce / taleggio
7,Mexican,salt / onions / ground cumin,ground cumin / chili powder / salt,avocado / jalapeno chilies / salsa,salt / onions / ground cumin,corn tortillas / salsa / avocado,ro-tel diced tomatoes & green chilies / chunky mild salsa / margarita salt
8,Chinese,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,soy sauce / sesame oil / corn starch,frozen popcorn chicken / oyster-flavor sauc / chopped cooked meat
9,British,salt / all-purpose flour / butter,milk / all-purpose flour / salt,milk / unsalted butter / all-purpose flour,salt / all-purpose flour / butter,unsalted butter / all-purpose flour / milk,meringue nests / chocolate covered english toffee / porridge oats


A quick scan confirms that 1-vs-all seems to produce the most distinctive keywords by far.

## Acknowledgements and References

Wendy Kan. What's Cooking?. https://www.kaggle.com/competitions/whats-cooking, 2015. Kaggle.